In [1]:
import pyarrow.parquet as pq
from pathlib import Path

arquivo = Path(
    "data/consumo/uf=PA/desempenho_grupos_municipal.parquet"
)

pf = pq.ParquetFile(arquivo)
meta = pf.metadata

print(f"Tamanho: {arquivo.stat().st_size / 1024**2:.2f} MB")
print(f"Linhas: {meta.num_rows:,}")
print(f"Colunas: {meta.num_columns}")
print(f"Row groups: {meta.num_row_groups}")
print(f"Criado por: {meta.created_by}")

print("\nESTRUTURA DAS COLUNAS")
print(pf.schema)

print("\nDETALHAMENTO DOS ROW GROUPS")
for i in range(meta.num_row_groups):
    rg = meta.row_group(i)
    print(
        f"Grupo {i}: "
        f"linhas={rg.num_rows:,} | "
        f"tamanho={rg.total_byte_size / 1024**2:.2f} MB"
    )

Tamanho: 8.57 MB
Linhas: 337,214
Colunas: 16
Row groups: 1
Criado por: parquet-cpp-arrow version 25.0.0

ESTRUTURA DAS COLUNAS
required group field_id=-1 schema {
  optional int64 field_id=-1 ano;
  optional binary field_id=-1 CO_MUNICIPIO_TERRITORIO (String);
  optional binary field_id=-1 NO_MUNICIPIO_TERRITORIO (String);
  optional binary field_id=-1 TIPO_REFERENCIA_TERRITORIAL (String);
  optional binary field_id=-1 tipo_grupo (String);
  optional binary field_id=-1 grupo (String);
  optional binary field_id=-1 disciplina (String);
  optional int64 field_id=-1 n_validos;
  optional int64 field_id=-1 n_zeros;
  optional double field_id=-1 media;
  optional double field_id=-1 media_positivas;
  optional double field_id=-1 mediana;
  optional double field_id=-1 std;
  optional double field_id=-1 q1;
  optional double field_id=-1 q3;
  optional binary field_id=-1 base_origem (String);
}


DETALHAMENTO DOS ROW GROUPS
Grupo 0: linhas=337,214 | tamanho=10.34 MB


In [2]:
import pyarrow.parquet as pq

arquivo = "data/consumo/uf=PA/desempenho_grupos_municipal.parquet"

pf = pq.ParquetFile(arquivo)
rg = pf.metadata.row_group(0)

for i in range(rg.num_columns):
    col = rg.column(i)

    print(
        f"{col.path_in_schema:35} "
        f"| compressão={col.compression:12} "
        f"| codificação={col.encodings} "
        f"| comprimido={col.total_compressed_size / 1024:.1f} KB "
        f"| descomprimido={col.total_uncompressed_size / 1024:.1f} KB"
    )

ano                                 | compressão=ZSTD         | codificação=('PLAIN', 'RLE', 'RLE_DICTIONARY') | comprimido=1.6 KB | descomprimido=1.6 KB
CO_MUNICIPIO_TERRITORIO             | compressão=ZSTD         | codificação=('PLAIN', 'RLE', 'RLE_DICTIONARY') | comprimido=20.4 KB | descomprimido=31.0 KB
NO_MUNICIPIO_TERRITORIO             | compressão=ZSTD         | codificação=('PLAIN', 'RLE', 'RLE_DICTIONARY') | comprimido=25.5 KB | descomprimido=38.3 KB
TIPO_REFERENCIA_TERRITORIAL         | compressão=ZSTD         | codificação=('PLAIN', 'RLE', 'RLE_DICTIONARY') | comprimido=1.1 KB | descomprimido=1.0 KB
tipo_grupo                          | compressão=ZSTD         | codificação=('PLAIN', 'RLE', 'RLE_DICTIONARY') | comprimido=48.8 KB | descomprimido=84.9 KB
grupo                               | compressão=ZSTD         | codificação=('PLAIN', 'RLE', 'RLE_DICTIONARY') | comprimido=201.5 KB | descomprimido=207.0 KB
disciplina                          | compressão=ZSTD         | co

##3. Próximo teste: separar PyArrow de Pandas

In [3]:
import pyarrow.parquet as pq
import psutil
import os
import gc

arquivo = "data/consumo/uf=PA/desempenho_grupos_municipal.parquet"

processo = psutil.Process(os.getpid())

def memoria():
    m = processo.memory_full_info()
    return {
        "RSS": m.rss / 1024**2,
        "USS": m.uss / 1024**2
    }

print("Inicial:", memoria())

# Leitura Arrow
tabela = pq.read_table(arquivo)

print("Após leitura Arrow:", memoria())

# Conversão para Pandas
df = tabela.to_pandas()

print("Após conversão Pandas:", memoria())

print("\nMemória DataFrame:")
print(f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nTipos:")
print(df.dtypes)

del df, tabela
gc.collect()

print("\nApós liberar objetos:", memoria())

Inicial: {'RSS': 104.90234375, 'USS': 53.8203125}
Após leitura Arrow: {'RSS': 307.9296875, 'USS': 237.0703125}
Após conversão Pandas: {'RSS': 336.0546875, 'USS': 264.3125}

Memória DataFrame:
55.47 MB

Tipos:
ano                              Int64
CO_MUNICIPIO_TERRITORIO         string
NO_MUNICIPIO_TERRITORIO            str
TIPO_REFERENCIA_TERRITORIAL        str
tipo_grupo                         str
grupo                              str
disciplina                         str
n_validos                        int64
n_zeros                          int64
media                          float64
media_positivas                float64
mediana                        float64
std                            float64
q1                             float64
q3                             float64
base_origem                        str
dtype: object

Após liberar objetos: {'RSS': 333.90234375, 'USS': 261.79296875}


In [4]:
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
import os

arquivo = "data/consumo/uf=PA/desempenho_grupos_municipal.parquet"

processo = psutil.Process(os.getpid())
pool = pa.default_memory_pool()

def diagnostico(etapa, tabela=None):
    memoria = processo.memory_full_info()

    print(f"\n{etapa}")
    print(f"RSS: {memoria.rss / 1024**2:.2f} MB")
    print(f"USS: {memoria.uss / 1024**2:.2f} MB")
    print(f"PyArrow pool: {pool.bytes_allocated() / 1024**2:.2f} MB")
    print(f"PyArrow pico: {pool.max_memory() / 1024**2:.2f} MB")

    if tabela is not None:
        print(f"Buffers da tabela: {tabela.nbytes / 1024**2:.2f} MB")
        print(f"Tamanho dos buffers: {tabela.get_total_buffer_size() / 1024**2:.2f} MB")


diagnostico("ANTES DA LEITURA")

tabela = pq.read_table(arquivo, use_threads=True)

diagnostico("LEITURA COM THREADS", tabela)


ANTES DA LEITURA
RSS: 334.64 MB
USS: 262.53 MB
PyArrow pool: 0.00 MB
PyArrow pico: 78.66 MB

LEITURA COM THREADS
RSS: 354.93 MB
USS: 282.82 MB
PyArrow pool: 55.51 MB
PyArrow pico: 83.12 MB
Buffers da tabela: 55.51 MB
Tamanho dos buffers: 55.51 MB


In [5]:
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
import os

arquivo = "data/consumo/uf=PA/desempenho_grupos_municipal.parquet"

processo = psutil.Process(os.getpid())
pool = pa.default_memory_pool()

def diagnostico(etapa, tabela=None):
    memoria = processo.memory_full_info()

    print(f"\n{etapa}")
    print(f"RSS: {memoria.rss / 1024**2:.2f} MB")
    print(f"USS: {memoria.uss / 1024**2:.2f} MB")
    print(f"PyArrow pool: {pool.bytes_allocated() / 1024**2:.2f} MB")
    print(f"PyArrow pico: {pool.max_memory() / 1024**2:.2f} MB")

    if tabela is not None:
        print(f"Buffers da tabela: {tabela.nbytes / 1024**2:.2f} MB")
        print(f"Tamanho dos buffers: {tabela.get_total_buffer_size() / 1024**2:.2f} MB")


diagnostico("ANTES DA LEITURA")

tabela = pq.read_table(arquivo, use_threads=False)
diagnostico("LEITURA SEM THREADS", tabela)


ANTES DA LEITURA
RSS: 354.95 MB
USS: 282.82 MB
PyArrow pool: 55.51 MB
PyArrow pico: 83.12 MB

LEITURA SEM THREADS
RSS: 431.04 MB
USS: 358.93 MB
PyArrow pool: 55.51 MB
PyArrow pico: 143.76 MB
Buffers da tabela: 55.51 MB
Tamanho dos buffers: 55.51 MB
